In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import csv
import os

os.makedirs("dataset/images", exist_ok=True)

with open("dataset/labels.csv", mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["filename", "a", "b", "c", "d", "e", "f"])

    saved_count = 0

    for i in range(5000):

        # --- random coefficients (fully free) ---
        a, b, c = np.random.uniform(-1, 1, 3)
        d, e = np.random.uniform(-2, 2, 2)
        f0 = np.random.uniform(-3, 3)

        # avoid nearly degenerate quadratic part
        if abs(a) + abs(b) + abs(c) < 0.25:
            continue

        # normalize to remove scale ambiguity
        norm = np.sqrt(a*a + b*b + c*c + d*d + e*e + f0*f0)
        a, b, c, d, e, f0 = a/norm, b/norm, c/norm, d/norm, e/norm, f0/norm

        # --- grid definition ---
        x = np.linspace(-10, 10, 400)
        y = np.linspace(-10, 10, 400)
        X, Y = np.meshgrid(x, y)
        Z = a*X**2 + b*X*Y + c*Y**2 + d*X + e*Y + f0

        # ensure the curve actually exists in the grid
        if not (np.any(Z > 0) and np.any(Z < 0)):
            continue

        # --- plot curve ---
        fig, ax = plt.subplots(figsize=(3, 3))
        ax.contour(X, Y, Z, levels=[0], colors='black')
        ax.axis('off')
        ax.set_aspect('equal')

        filename = f"sample_{saved_count:04d}.png"
        plt.savefig(
            f"dataset/images/{filename}",
            dpi=100,
            bbox_inches='tight',
            pad_inches=0
        )
        plt.close(fig)

        writer.writerow([filename, a, b, c, d, e, f0])
        saved_count += 1

In [ ]:
import tensorflow as tf
import os

IMG_DIR = "dataset/images"

def load_sample(row):
    img_path = os.path.join(IMG_DIR, row["filename"])
    
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, (224, 224))
    img = tf.cast(img, tf.float32) / 255.0

    coeffs = tf.constant(
        [row["a"], row["b"], row["c"], row["d"], row["e"], row["f"]],
        dtype=tf.float32
    )

    return img, coeffs

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

inputs = layers.Input(shape=(224, 224, 3))

x = inputs

# --- shared CNN backbone ---
for filters in [8, 16, 32, 64, 128]:
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D(2)(x)

# Aggregate spatial information without exploding parameter count
x = layers.GlobalAveragePooling2D()(x)

# Small intermediate dense layer for more stable regression
x = layers.Dense(128, activation="relu")(x)

# --- output head: 6 quadratic coefficients ---
coeffs = layers.Dense(6, activation="linear")(x)

model = Model(inputs=inputs, outputs=coeffs)

In [ ]:
import tensorflow as tf

def sign_invariant_mse(y_true, y_pred):
    loss1 = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)
    loss2 = tf.reduce_mean(tf.square(y_true + y_pred), axis=-1)
    return tf.minimum(loss1, loss2)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=sign_invariant_mse,
    metrics=["mae"]
)

In [ ]:
import pandas as pd

df = pd.read_csv("dataset/labels.csv")

In [ ]:
def dataset_from_dataframe(df, batch_size=16, shuffle=True):

    def gen():
        for _, row in df.iterrows():
            yield load_sample(row)

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=(224, 224, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(6,), dtype=tf.float32)
        )
    )

    # Shuffle the entire dataset if requested
    if shuffle:
        ds = ds.shuffle(len(df))

    # Create batches and enable asynchronous prefetching
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

In [ ]:
from sklearn.model_selection import train_test_split

# Split the dataset into training and validation sets
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

# Reset indices after splitting for clean iteration
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

# Create TensorFlow datasets
train_ds = dataset_from_dataframe(train_df, batch_size=16, shuffle=True)
val_ds = dataset_from_dataframe(val_df, batch_size=16, shuffle=False)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=callbacks
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

save_dir = "random_test"
os.makedirs(save_dir, exist_ok=True)

while True:

    # --- random quadratic coefficients (fully free) ---
    a, b, c = np.random.uniform(-1, 1, 3)
    d, e = np.random.uniform(-2, 2, 2)
    f = np.random.uniform(-3, 3)

    # Avoid nearly degenerate quadratic part
    if abs(a) + abs(b) + abs(c) < 0.25:
        continue

    # Normalize to remove scale ambiguity (same scaling as training data)
    norm = np.sqrt(a*a + b*b + c*c + d*d + e*e + f*f)
    a, b, c, d, e, f = a/norm, b/norm, c/norm, d/norm, e/norm, f/norm

    # Generate evaluation grid
    x = np.linspace(-10, 10, 400)
    y = np.linspace(-10, 10, 400)
    X, Y = np.meshgrid(x, y)
    Z = a*X**2 + b*X*Y + c*Y**2 + d*X + e*Y + f

    # Ensure the curve actually exists within the grid
    if np.any(Z > 0) and np.any(Z < 0):
        break

# Plot the implicit curve (level set Z=0)
plt.figure(figsize=(3, 3))
plt.contour(X, Y, Z, levels=[0], colors='black')
plt.axis('off')
plt.gca().set_aspect('equal')

img_path = os.path.join(save_dir, "random_curve.png")
plt.savefig(img_path, dpi=100, bbox_inches='tight', pad_inches=0)
plt.close()

print("True equation:")
print(f"{a:.3f}x² + {b:.3f}xy + {c:.3f}y² + {d:.3f}x + {e:.3f}y + {f:.3f} = 0")

In [ ]:
from tensorflow.keras.utils import load_img, img_to_array
import numpy as np

def predict_equation(img_path, model):

    # Preprocess the image (consistent with training pipeline)
    img = load_img(img_path, target_size=(224, 224))
    img = img_to_array(img) / 255.0
    img = np.expand_dims(img, axis=0)

    # Predict quadratic coefficients
    coeffs_pred = model.predict(img, verbose=0)[0]

    # Normalize prediction to match training scale convention
    norm = np.linalg.norm(coeffs_pred)
    coeffs_pred = coeffs_pred / norm

    a, b, c, d, e, f = coeffs_pred

    eq = (
        f"{a:.3f} x² + "
        f"{b:.3f} xy + "
        f"{c:.3f} y² + "
        f"{d:.3f} x + "
        f"{e:.3f} y + "
        f"{f:.3f} = 0"
    )

    return coeffs_pred, eq

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- prediction ---
pred_coeffs, pred_eq = predict_equation(img_path, model)

# sign alignment
true_coeffs = np.array([a, b, c, d, e, f])
if np.dot(pred_coeffs, true_coeffs) < 0:
    pred_coeffs = -pred_coeffs

a_p, b_p, c_p, d_p, e_p, f_p = pred_coeffs

# --- grid ---
x = np.linspace(-10, 10, 400)
y = np.linspace(-10, 10, 400)
X, Y = np.meshgrid(x, y)

Z_true = a*X**2 + b*X*Y + c*Y**2 + d*X + e*Y + f
Z_pred = a_p*X**2 + b_p*X*Y + c_p*Y**2 + d_p*X + e_p*Y + f_p

# --- plot ---
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].contour(X, Y, Z_true, levels=[0], colors='black')
axes[0].set_title("True")
axes[0].set_aspect('equal')
axes[0].axis('off')

axes[1].contour(X, Y, Z_pred, levels=[0], colors='red')
axes[1].set_title("Predicted")
axes[1].set_aspect('equal')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
test_dir = "random_test_set"
os.makedirs(test_dir, exist_ok=True)

test_coeffs = []

for i in range(100):

    while True:
        a, b, c = np.random.uniform(-1, 1, 3)
        d, e = np.random.uniform(-2, 2, 2)
        f = np.random.uniform(-3, 3)

        if abs(a) + abs(b) + abs(c) < 0.25:
            continue

        norm = np.sqrt(a*a + b*b + c*c + d*d + e*e + f*f)
        a, b, c, d, e, f = a/norm, b/norm, c/norm, d/norm, e/norm, f/norm

        x = np.linspace(-10, 10, 400)
        y = np.linspace(-10, 10, 400)
        X, Y = np.meshgrid(x, y)
        Z = a*X**2 + b*X*Y + c*Y**2 + d*X + e*Y + f

        if np.any(Z > 0) and np.any(Z < 0):
            break

    plt.figure(figsize=(3,3))
    plt.contour(X, Y, Z, levels=[0], colors='black')
    plt.axis('off')
    plt.gca().set_aspect('equal')

    img_path = os.path.join(test_dir, f"test_{i}.png")
    plt.savefig(img_path, dpi=100, bbox_inches='tight', pad_inches=0)
    plt.close()

    test_coeffs.append([a,b,c,d,e,f])

In [ ]:
errors = []

for i in range(100):

    img_path = os.path.join(test_dir, f"test_{i}.png")
    pred_coeffs, _ = predict_equation(img_path, model)

    true = np.array(test_coeffs[i])

    # sign alignment
    if np.dot(pred_coeffs, true) < 0:
        pred_coeffs = -pred_coeffs

    err = np.linalg.norm(pred_coeffs - true)
    errors.append(err)

print("Mean L2 error:", np.mean(errors))
print("Max L2 error:", np.max(errors))

In [ ]:
idx = np.argmax(errors)
print("Worst index:", idx)
print("Error:", errors[idx])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

# worst index
idx = np.argmax(errors)
print("Worst index:", idx)
print("L2 error:", errors[idx])

# true coeffs
true = np.array(test_coeffs[idx])

# prediction
img_path = os.path.join(test_dir, f"test_{idx}.png")
pred_coeffs, _ = predict_equation(img_path, model)

# sign alignment
if np.dot(pred_coeffs, true) < 0:
    pred_coeffs = -pred_coeffs

# grid
x = np.linspace(-10, 10, 400)
y = np.linspace(-10, 10, 400)
X, Y = np.meshgrid(x, y)

Z_true = (
    true[0]*X**2 + true[1]*X*Y + true[2]*Y**2 +
    true[3]*X + true[4]*Y + true[5]
)

Z_pred = (
    pred_coeffs[0]*X**2 + pred_coeffs[1]*X*Y + pred_coeffs[2]*Y**2 +
    pred_coeffs[3]*X + pred_coeffs[4]*Y + pred_coeffs[5]
)

# plot
plt.figure(figsize=(5,5))
plt.contour(X, Y, Z_true, levels=[0], colors='black')
plt.contour(X, Y, Z_pred, levels=[0], colors='red')
plt.gca().set_aspect('equal')
plt.axis('off')
plt.title("Black = True | Red = Predicted")
plt.show()

In [ ]:
# --- SUCCESS EXAMPLE EXPORT ---

import numpy as np
import matplotlib.pyplot as plt
import os

export_dir = "learning_based/quadratic/coefficient_loss/assets"
os.makedirs(export_dir, exist_ok=True)

# Generate a fresh random curve
while True:
    a, b, c = np.random.uniform(-1, 1, 3)
    d, e = np.random.uniform(-2, 2, 2)
    f = np.random.uniform(-3, 3)

    if abs(a) + abs(b) + abs(c) < 0.25:
        continue

    norm = np.sqrt(a*a + b*b + c*c + d*d + e*e + f*f)
    a, b, c, d, e, f = a/norm, b/norm, c/norm, d/norm, e/norm, f/norm

    x = np.linspace(-10, 10, 400)
    y = np.linspace(-10, 10, 400)
    X, Y = np.meshgrid(x, y)
    Z = a*X**2 + b*X*Y + c*Y**2 + d*X + e*Y + f

    if np.any(Z > 0) and np.any(Z < 0):
        break

# Save input image
plt.figure(figsize=(3,3))
plt.contour(X, Y, Z, levels=[0], colors='black')
plt.axis('off')
plt.gca().set_aspect('equal')

success_input_path = os.path.join(export_dir, "example_success.png")
plt.savefig(success_input_path, dpi=250, bbox_inches='tight', pad_inches=0)
plt.close()

# Prediction
pred_coeffs, _ = predict_equation(success_input_path, model)

true = np.array([a,b,c,d,e,f])

if np.dot(pred_coeffs, true) < 0:
    pred_coeffs = -pred_coeffs

# Overlay figure
Z_true = a*X**2 + b*X*Y + c*Y**2 + d*X + e*Y + f
Z_pred = (
    pred_coeffs[0]*X**2 + pred_coeffs[1]*X*Y + pred_coeffs[2]*Y**2 +
    pred_coeffs[3]*X + pred_coeffs[4]*Y + pred_coeffs[5]
)

plt.figure(figsize=(4,4))
plt.contour(X, Y, Z_true, levels=[0], colors='black')
plt.contour(X, Y, Z_pred, levels=[0], colors='red')
plt.axis('off')
plt.gca().set_aspect('equal')

overlay_path = os.path.join(export_dir, "example_success_overlay.png")
plt.savefig(overlay_path, dpi=250, bbox_inches='tight', pad_inches=0)
plt.close()


In [ ]:
# --- WORST CASE EXPORT ---

idx = np.argmax(errors)
true = np.array(test_coeffs[idx])

img_path = os.path.join(test_dir, f"test_{idx}.png")
pred_coeffs, _ = predict_equation(img_path, model)

if np.dot(pred_coeffs, true) < 0:
    pred_coeffs = -pred_coeffs

x = np.linspace(-10, 10, 400)
y = np.linspace(-10, 10, 400)
X, Y = np.meshgrid(x, y)

Z_true = (
    true[0]*X**2 + true[1]*X*Y + true[2]*Y**2 +
    true[3]*X + true[4]*Y + true[5]
)

Z_pred = (
    pred_coeffs[0]*X**2 + pred_coeffs[1]*X*Y + pred_coeffs[2]*Y**2 +
    pred_coeffs[3]*X + pred_coeffs[4]*Y + pred_coeffs[5]
)

plt.figure(figsize=(4,4))
plt.contour(X, Y, Z_true, levels=[0], colors='black')
plt.contour(X, Y, Z_pred, levels=[0], colors='red')
plt.axis('off')
plt.gca().set_aspect('equal')

failure_path = os.path.join(export_dir, "example_failure_overlay.png")
plt.savefig(failure_path, dpi=250, bbox_inches='tight', pad_inches=0)
plt.close()
